# 올리브영 랭킹 알고리즘 추정 — 스킨케어 > 에센스/세럼/앰플

**분석 질문**: 검색 결과 상위 노출("인기순")은 리뷰수·평점·가격 중 무엇과 관련 있는가?

**대상 페이지**: 올리브영 스킨케어 > 에센스/세럼/앰플 카테고리, 인기순 정렬, 상위 50개
`https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=1000001000100140001&prdSort=01`

**수집 시점**: 2026-07-23

**절차**
1. 카테고리 목록 페이지에서 순위·브랜드·상품명·가격·평점(추정)·goodsNo를 수집한다.
2. 목록 페이지는 리뷰수를 `999+`로 뭉개서 보여주므로, 각 상품 상세페이지를 헤드리스 브라우저로 직접 렌더링해 **정확한 리뷰수**를 다시 수집한다.
3. 순위와 각 지표(리뷰수/평점/가격/할인율) 사이의 스피어만·피어슨 상관계수를 계산한다.
4. 산점도로 시각화하고 결론을 정리한다.

## 0. 환경 준비

In [ ]:
# 최초 1회만 실행
# !pip install curl_cffi beautifulsoup4 lxml playwright pandas scipy matplotlib
# !playwright install chromium


In [ ]:
import re
import time
import json
import pandas as pd
import numpy as np
from scipy import stats
from bs4 import BeautifulSoup
from curl_cffi import requests as cf_requests


## 1. 카테고리 목록 페이지 수집

`www.oliveyoung.co.kr`은 일반 `requests`로는 403이 뜨는 경우가 있어, TLS 지문을 흉내내는
`curl_cffi`(`impersonate="safari"`)를 사용한다. `rowsPerPage=50`으로 요청하면 인기순 상위 50개를
한 페이지에서 받아올 수 있다.

In [ ]:
LIST_URL = (
    "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do"
    "?dispCatNo=1000001000100140001&fltDispCatNo=&prdSort=01"
    "&pageIdx=1&rowsPerPage=50&searchTypeSort=btn_thumb"
    "&plusButtonFlag=N&isLoginCnt=&trackingCd=Cat1000001000100140001_Small"
)

resp = cf_requests.get(
    LIST_URL,
    impersonate="safari",
    headers={"Referer": "https://www.oliveyoung.co.kr/"},
    timeout=25,
)
print(resp.status_code, len(resp.text))
list_html = resp.text


## 2. 목록 페이지 파싱

상품 리스트는 `<li data-number="N">` 블록 단위로 반복된다. `lxml` 파서에 전체 HTML을 통째로
넣으면 5번째 상품(베스트 리본이 있는 항목)부터 트리가 깨지는 문제가 있어, 정규식으로 각 `<li>`
블록을 먼저 잘라낸 뒤 블록별로 개별 파싱한다.

평점은 화면에 보이는 "10점만점에 5.5점(999+)" 텍스트가 실제 값이 아니라 자바스크립트가 채우기
전의 스크린리더용 placeholder다. 실제 평점은 별점 바의 `style="width:96.0%"` 값이며,
`퍼센트 ÷ 20 = 5점 만점 평점`이다 (96% → 4.8점).

In [ ]:
def parse_list_page(html, max_rank=50):
    starts = [m.start() for m in re.finditer(r'<li criteo-goods="[^"]*"[^>]*data-number="\d+">', html)]
    starts.append(len(html))

    rows = []
    for i in range(len(starts) - 1):
        frag = html[starts[i]:starts[i + 1]]
        m = re.search(r'data-number="(\d+)"', frag)
        rank = int(m.group(1))
        if rank > max_rank:
            continue

        goods_m = re.search(r'data-ref-goodsNo="([^"]+)"', frag)
        goods_no = goods_m.group(1) if goods_m else None

        soup = BeautifulSoup(frag, "lxml")
        brand = (soup.select_one(".tx_brand") or {}).get_text(strip=True) if soup.select_one(".tx_brand") else ""
        name = soup.select_one(".tx_name").get_text(strip=True) if soup.select_one(".tx_name") else ""

        cur_tag = soup.select_one(".prd_price .tx_cur .tx_num")
        org_tag = soup.select_one(".prd_price .tx_org .tx_num")
        price = cur_tag.get_text(strip=True) if cur_tag else (soup.select_one(".prd_price .tx_num") or {}).get_text(strip=True)
        org_price = org_tag.get_text(strip=True) if org_tag else None

        point_tag = soup.select_one(".review_point .point")
        rating = None
        if point_tag and point_tag.get("style"):
            wm = re.search(r"width\s*:\s*([\d.]+)%", point_tag["style"])
            if wm:
                rating = round(float(wm.group(1)) / 20, 1)

        point_area = soup.select_one(".prd_point_area")
        review_capped = None
        if point_area:
            cm = re.search(r"\(([^)]+)\)", point_area.get_text(" ", strip=True))
            if cm:
                review_capped = cm.group(1)

        rows.append({
            "rank": rank, "goods_no": goods_no, "brand": brand, "name": name,
            "price": price, "org_price": org_price, "rating": rating,
            "review_count_capped": review_capped,
        })

    rows.sort(key=lambda r: r["rank"])
    return rows


listing = parse_list_page(list_html, max_rank=50)
print(len(listing), "products parsed")
listing[0]


## 3. 상세페이지에서 정확한 리뷰수 수집

상품 상세페이지(`getGoodsDetail.do`)는 Next.js로 렌더링되며, 리뷰수는 서버 HTML에 없고
클라이언트에서 API를 호출한 뒤 하이드레이션되며 채워진다. 정적 HTML 파싱으로는 얻을 수 없어
Playwright로 실제 페이지를 렌더링해 텍스트에서 "평점 / 리뷰 N건" 패턴을 읽는다.

In [ ]:
from playwright.sync_api import sync_playwright

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
      "(KHTML, like Gecko) Version/17.4 Safari/605.1.15")

rating_re = re.compile(r"평점\s*\n?\s*([\d.]+)\s*\n\s*리뷰\s*([\d,]+)\s*건")
review_only_re = re.compile(r"리뷰\s*([\d,]+)\s*건")


def fetch_exact_review(page, goods_no, retries=2):
    url = f"https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo={goods_no}"
    for attempt in range(retries + 1):
        page.goto(url, wait_until="domcontentloaded", timeout=30000)
        page.wait_for_timeout(2500 if attempt == 0 else 4000)
        text = page.inner_text("body")
        m = rating_re.search(text)
        if m:
            return float(m.group(1)), int(m.group(2).replace(",", ""))
        m2 = review_only_re.search(text)
        if m2:
            return None, int(m2.group(1).replace(",", ""))
    return None, None


results = []
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    context = browser.new_context(user_agent=UA, viewport={"width": 1280, "height": 2000}, locale="ko-KR")
    page = context.new_page()

    for row in listing:
        exact_rating, exact_review = fetch_exact_review(page, row["goods_no"])
        row["exact_rating"] = exact_rating if exact_rating is not None else row["rating"]
        row["exact_review_count"] = exact_review
        results.append(row)
        print(row["rank"], row["goods_no"], row["exact_rating"], row["exact_review_count"])
        time.sleep(0.6)

    browser.close()


## 4. 데이터프레임 정리 및 파생 변수 계산

In [ ]:
df = pd.DataFrame(results)
df["price_num"] = df["price"].str.replace(",", "").astype(int)
df["org_price_num"] = df["org_price"].apply(lambda x: int(x.replace(",", "")) if x else None)
df["discount_pct"] = (1 - df["price_num"] / df["org_price_num"]) * 100

df.to_csv("oliveyoung_top50_essence_serum.csv", index=False, encoding="utf-8-sig")
df.head(10)


## 5. 상관관계 분석

"인기순위"(1위가 가장 높은 노출)와 각 지표 사이의 **스피어만 순위상관계수(ρ)**와
**피어슨 상관계수(r)**를 계산한다. ρ가 0에 가까울수록 두 값이 무관하다는 뜻이다.

In [ ]:
def correlate(df, col):
    sub = df.dropna(subset=[col])
    rho, p_s = stats.spearmanr(sub["rank"], sub[col])
    r, p_p = stats.pearsonr(sub["rank"], sub[col])
    return {"rho": round(rho, 3), "spearman_p": round(p_s, 4),
            "r": round(r, 3), "pearson_p": round(p_p, 4), "n": len(sub)}

summary = {
    "리뷰수": correlate(df, "exact_review_count"),
    "평점": correlate(df, "exact_rating"),
    "판매가": correlate(df, "price_num"),
    "할인율": correlate(df, "discount_pct"),
}
pd.DataFrame(summary).T


## 6. 상위 10위 vs 하위 10위 비교

In [ ]:
top10 = df[df["rank"] <= 10]
bottom10 = df[df["rank"] > 40]

comparison = pd.DataFrame({
    "상위10 (1-10위)": [
        top10["exact_review_count"].median(),
        top10["exact_rating"].mean(),
        top10["price_num"].mean(),
    ],
    "하위10 (41-50위)": [
        bottom10["exact_review_count"].median(),
        bottom10["exact_rating"].mean(),
        bottom10["price_num"].mean(),
    ],
}, index=["리뷰수(중앙값)", "평점(평균)", "판매가(평균)"])
comparison


## 7. 시각화 — 순위 vs 지표 산점도

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
# 한글 폰트가 필요하면 아래 주석을 해제하고 환경에 맞는 폰트로 바꿔서 사용
# plt.rcParams["font.family"] = "AppleGothic"  # macOS
# plt.rcParams["font.family"] = "Malgun Gothic"  # Windows

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

def scatter_with_trend(ax, y_col, ylabel, log=False):
    x = df["rank"]
    y = df[y_col]
    ax.scatter(x, y, alpha=0.7, edgecolor="white", s=40)
    yy = np.log10(y) if log else y
    slope, intercept = np.polyfit(x, yy, 1)
    xs = np.linspace(1, 50, 100)
    trend = slope * xs + intercept
    ax.plot(xs, 10**trend if log else trend, "--", color="darkorange", linewidth=1.5)
    if log:
        ax.set_yscale("log")
    ax.set_xlabel("인기순위 (1→50)")
    ax.set_ylabel(ylabel)

scatter_with_trend(axes[0, 0], "exact_review_count", "리뷰수 (log)", log=True)
scatter_with_trend(axes[0, 1], "exact_rating", "평점")
scatter_with_trend(axes[1, 0], "price_num", "판매가(원)")
scatter_with_trend(axes[1, 1], "discount_pct", "할인율(%)")

fig.suptitle("올리브영 인기순위 vs 리뷰수·평점·가격·할인율 (에센스/세럼/앰플 TOP 50)")
fig.tight_layout()
plt.savefig("oliveyoung_ranking_scatter.png", dpi=150)
plt.show()


## 8. 결론

상위 50개 제품 기준으로, **리뷰수·평점·가격·할인율 중 어느 것도 "인기순" 노출 순위와
통계적으로 유의한 상관관계를 보이지 않았다** (모든 p > 0.05). 상위 10위와 하위 10위 그룹을
직접 비교해도 리뷰수·평점·가격에 뚜렷한 차이가 없었다.

이는 올리브영의 "인기순" 정렬이 화면에 노출되는 정적 지표(누적 리뷰수, 평균 평점, 가격)보다
**최근 판매량, 최근 클릭/장바구니 추가율, 프로모션·광고 편성 여부** 등 비공개 신호에 더 크게
좌우될 가능성을 시사한다. 다만 본 분석은 특정 시점의 스냅샷(n=50)이므로 인과관계를 증명하지
않으며, 표본을 늘리거나(예: 200개), 시계열로 반복 수집하면 더 신뢰도 높은 결론을 얻을 수 있다.